In [ ]:
import pandas as pd

DATA_PATH = "health_synthetic_200users_90days.csv"

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.info()
df.describe()
df.isna().sum()


In [ ]:
df = df.sort_values(["user_id", "day"]).reset_index(drop=True)


In [ ]:
def load_health_data(path="health_synthetic_200users_90days.csv"):
    df = pd.read_csv(path)
    df = df.sort_values(["user_id", "day"]).reset_index(drop=True)
    return df

df = load_health_data()
df.head()



In [ ]:
df[["steps", "sleep_hours", "stress_level", "fatigue_score"]].describe()


In [ ]:
import matplotlib.pyplot as plt

df["steps"].hist(bins=30)
plt.title("Steps distribution")

df["sleep_hours"].hist(bins=20)
plt.title("Sleep Hours distribution")


## Feature engineering (rolling averages + flags)

In [ ]:
df_feat = df.copy()

# 7-day rolling averages per user
df_feat["avg_steps_7d"] = (
    df_feat.groupby("user_id")["steps"]
    .rolling(7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df_feat["avg_sleep_7d"] = (
    df_feat.groupby("user_id")["sleep_hours"]
    .rolling(7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df_feat["avg_stress_7d"] = (
    df_feat.groupby("user_id")["stress_level"]
    .rolling(7, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# Flags for current day
df_feat["low_sleep_flag"]   = (df_feat["sleep_hours"] < 6).astype(int)
df_feat["high_stress_flag"] = (df_feat["stress_level"] >= 7).astype(int)
df_feat["low_steps_flag"]   = (df_feat["steps"] < 6000).astype(int)
df_feat["low_water_flag"]   = (df_feat["water_glasses"] < 6).astype(int)
df_feat["high_fatigue_flag"] = (df_feat["fatigue_score"] >= 7).astype(int)

df_feat.head()


## Prepare data for the ML model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [ ]:
# Target already created above: high_fatigue_flag

feature_cols = [
    "avg_steps_7d",
    "avg_sleep_7d",
    "avg_stress_7d",
    "water_glasses",
    "calories_intake",
    "resting_heart_rate",
    "age",
    "bmi",
]

# Drop rows where rolling averages might be NaN (first few days)
ml_df = df_feat.dropna(subset=feature_cols).copy()

X = ml_df[feature_cols]
y = ml_df["high_fatigue_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

print("Train accuracy:", rf.score(X_train, y_train))
print("Test accuracy:", rf.score(X_test, y_test))


In [ ]:
def get_user_today(df_features, user_id):
    """
    Return the latest record (max day) for a given user_id.
    """
    user_df = df_features[df_features["user_id"] == user_id]
    if user_df.empty:
        raise ValueError(f"No data found for user_id={user_id}")
    return user_df.sort_values("day").iloc[-1]


In [ ]:
def recommend_actions(row, model=None):
    """
    row: single row (Series) from df_feat representing one user on one day.
    model: trained ML model (e.g. RandomForest) or None.
    
    returns:
      - recs (list of recommendation strings)
      - ml_risk (float or None): predicted probability of high fatigue
    """
    recs = []

    # --- Optional ML prediction ---
    ml_risk = None
    if model is not None:
        feats = row[feature_cols].values.reshape(1, -1)
        prob = model.predict_proba(feats)[0][1]  # P(high fatigue = 1)
        ml_risk = prob

    # Activity-related recommendations
    if row["avg_steps_7d"] < 6000:
        recs.append(
            "Increase your daily steps by 1,000–2,000 over the next week "
            "(for example, add a 15–20 minute walk)."
        )

    # Sleep-related recommendations
    if row["avg_sleep_7d"] < 7:
        recs.append(
            "Aim for at least 7 hours of sleep. Try going to bed 30 minutes earlier tonight."
        )

    # Hydration recommendations
    if row["water_glasses"] < 6:
        recs.append(
            "Increase your water intake to at least 6–8 glasses per day."
        )

    # Stress management
    if row["stress_level"] >= 7 or row["avg_stress_7d"] >= 7:
        recs.append(
            "Your stress levels are high. Try 10 minutes of relaxation, "
            "such as deep breathing, a walk, or a short mindfulness break."
        )

    # Rule-based fatigue
    if row["fatigue_score"] >= 7:
        recs.append(
            "You seem fatigued. Reduce workout intensity today and prioritize rest and light movement."
        )

    # ML-based fatigue risk (even if current score isn't high yet)
    if ml_risk is not None and ml_risk >= 0.7:
        recs.append(
            "Based on your recent patterns, you are at high risk of fatigue. "
            "Plan at least one lighter day this week and focus on sleep and stress management."
        )

    # If no specific issues, maintenance recommendation
    if not recs:
        recs.append(
            "You are on track today. Maintain your current routine and keep monitoring your sleep, activity, and stress."
        )

    return recs, ml_risk


In [ ]:
sample_row = get_user_today(df_feat, user_id=5)
recs, risk = recommend_actions(sample_row, model=rf)

print("User 5 - latest day")
print("Predicted high fatigue risk probability:", round(risk, 2))
print("\nRecommendations:")
for r in recs:
    print("-", r)


In [ ]:
pip install pdfplumber


In [ ]:
import pdfplumber
from pathlib import Path

RAW_KB_DIR = Path("./kb/raw_pdfs")
TEXT_OUTPUT_DIR = Path("./kb/processed")
TEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_text_from_pdf(pdf_path: Path) -> str:
    text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text.append(page_text)
    return "\n".join(text)

for pdf_file in RAW_KB_DIR.glob("*.pdf"):
    txt = extract_text_from_pdf(pdf_file)
    out_path = TEXT_OUTPUT_DIR / (pdf_file.stem + ".txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(txt)
    print("Processed:", pdf_file.name, "→", out_path.name)


In [ ]:
def split_into_chunks(text: str, max_chars: int = 800, overlap: int = 200):
    """
    Simple character-based chunking with overlap.
    """
    chunks = []
    start = 0
    length = len(text)

    while start < length:
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap  # move back a bit for overlap

        if start < 0:
            start = 0

    return [c for c in chunks if c]

import pandas as pd

all_chunks = []

for txt_file in TEXT_OUTPUT_DIR.glob("*.txt"):
    with open(txt_file, "r", encoding="utf-8") as f:
        full_text = f.read()
    chunks = split_into_chunks(full_text, max_chars=800, overlap=200)
    for i, ch in enumerate(chunks):
        all_chunks.append({
            "source": txt_file.stem,
            "chunk_id": f"{txt_file.stem}_{i}",
            "text": ch,
        })

kb_df = pd.DataFrame(all_chunks)
kb_df.head(), len(kb_df)

In [ ]:
KB_CHUNKS_PATH = TEXT_OUTPUT_DIR / "kb_chunks.parquet"
kb_df.to_parquet(KB_CHUNKS_PATH, index=False)

In [ ]:
pip install sentence-transformers


In [ ]:
import torch
print(torch.__version__)


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
model_emb = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model_emb.encode(
    kb_df["text"].tolist(),
    show_progress_bar=True
)

embeddings = np.array(embeddings)
kb_df["embedding"] = list(embeddings)


In [ ]:
KB_INDEX_PATH = TEXT_OUTPUT_DIR / "kb_index.parquet"
kb_df.to_parquet(KB_INDEX_PATH, index=False)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def load_kb_index(path=KB_INDEX_PATH):
    df = pd.read_parquet(path)
    df["embedding"] = df["embedding"].apply(lambda x: np.array(x))
    return df

kb_index = load_kb_index()

def retrieve_relevant_chunks(query: str, user_context: dict | None = None, top_k: int = 5):
    full_query = query
    if user_context is not None:
        ctx_parts = []
        if "age" in user_context:
            ctx_parts.append(f"age {user_context['age']}")
        if "bmi" in user_context:
            ctx_parts.append(f"BMI {user_context['bmi']:.1f}")
        if ctx_parts:
            full_query = query + " | " + ", ".join(ctx_parts)

    q_emb = model_emb.encode([full_query])
    chunk_embs = np.stack(kb_index["embedding"].values)
    sims = cosine_similarity(q_emb, chunk_embs)[0]

    top_idx = np.argsort(sims)[::-1][:top_k]
    return kb_index.iloc[top_idx].copy(), sims[top_idx]


In [ ]:
test_chunks, test_scores = retrieve_relevant_chunks(
    "how much physical activity should adults do per week?",
    user_context={"age": 30, "bmi": 27},
    top_k=3
)

test_chunks[["source", "text"]]


In [ ]:
import os
from langchain_groq import ChatGroq


In [ ]:
os.environ["GROQ_API_KEY"] = "GROQ_API"


In [ ]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.2,
)

In [ ]:
import json

def parse_user_question_to_features(user_question: str):
    """
    Use the LLM to extract numeric lifestyle features from a free-text question.
    Returns a dict with keys needed for ML and rules.
    """
    extraction_prompt = f"""
You are an information extraction assistant.

From the following user's message, extract the following fields if they are mentioned.
If a field is not mentioned, set it to null.

Fields (JSON keys) you must output:
- avg_steps_7d (float, average daily steps over last 7 days)
- avg_sleep_7d (float, average sleep hours over last 7 days)
- avg_stress_7d (float, average stress level 1-10)
- water_glasses (int, glasses of water per day)
- calories_intake (int, approximate daily kcal)
- resting_heart_rate (int, beats per minute)
- age (int, years)
- bmi (float)
- steps (int, today's steps if mentioned)
- sleep_hours (float, today's sleep if mentioned)
- stress_level (int, today's stress 1-10 if mentioned)
- fatigue_score (int, today's fatigue 1-10 if mentioned)

User message:
\"\"\"{user_question}\"\"\"


Return ONLY a valid JSON object, no explanation and no extra text.
"""

    response = llm.invoke(extraction_prompt)
    raw = getattr(response, "content", str(response)).strip()

    # Try to parse JSON (handle backticks etc)
    # Remove ```json ``` wrappers if model adds them
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:].strip()

    data = json.loads(raw)

    return data


In [ ]:
def health_coach_agent_free_text(user_question: str, model, top_k: int = 3):
    """
    Main agent for NEW users who only provide a natural-language description
    of their habits (no row in the dataset).

    1) Parse features from text (LLM extraction)
    2) Compute fatigue risk with ML if enough info
    3) Build simple recommendations from habits
    4) Retrieve guideline chunks (RAG)
    5) Ask LLM to combine everything into one answer
    """
    # 1) Extract structured features
    feats = parse_user_question_to_features(user_question)

    # 2) Try to compute ML fatigue risk
    can_run_ml = all(feats.get(col) is not None for col in feature_cols)
    ml_prob = None
    if can_run_ml:
        import numpy as np
        X_user = np.array([[feats[col] for col in feature_cols]])
        ml_prob = float(model.predict_proba(X_user)[0][1])  # probability high fatigue
    
    # 3) Simple rule-based recommendations using extracted features
    recs = []

    if feats.get("avg_steps_7d") is not None and feats["avg_steps_7d"] < 6000:
        recs.append(
            "Your average steps are below 6,000. Gradually increase daily steps by 1,000–2,000 over the next week."
        )

    if feats.get("avg_sleep_7d") is not None and feats["avg_sleep_7d"] < 7:
        recs.append(
            "Your average sleep is under 7 hours. Aim for a more regular sleep schedule and try to reach 7–9 hours."
        )

    if feats.get("avg_stress_7d") is not None and feats["avg_stress_7d"] >= 7:
        recs.append(
            "Your stress levels are high. Consider daily relaxation breaks, breathing exercises, or light walks."
        )

    if feats.get("water_glasses") is not None and feats["water_glasses"] < 6:
        recs.append(
            "Your water intake seems low. Aim for at least 6–8 glasses of water per day, unless advised otherwise."
        )

    if feats.get("resting_heart_rate") is not None and feats["resting_heart_rate"] > 80:
        recs.append(
            "Your resting heart rate is on the higher side. Increasing regular activity and managing stress may help."
        )

    if not recs:
        recs.append(
            "Your habits are partly on track. Focus on keeping consistency in activity, sleep, hydration and stress."
        )

    # 4) Build user_context for RAG retrieval
    user_context = {k: v for k, v in feats.items() if k in [
        "age", "bmi", "avg_steps_7d", "avg_sleep_7d", "avg_stress_7d",
        "water_glasses", "calories_intake", "resting_heart_rate"
    ]}
    user_context["predicted_high_fatigue_prob"] = ml_prob

    # 5) Retrieve RAG chunks based on the user question
    top_chunks, scores = retrieve_relevant_chunks(
        user_question,
        user_context=user_context,
        top_k=top_k
    )
    context_text = "\n\n".join(top_chunks["text"].tolist())

    # 6) Build final prompt that combines everything
    ml_risk_text = "not computed (missing data)"
    if ml_prob is not None:
        if ml_prob >= 0.7:
            lvl = "HIGH"
        elif ml_prob >= 0.4:
            lvl = "MODERATE"
        else:
            lvl = "LOW"
        ml_risk_text = f"{ml_prob:.2f} ({lvl} fatigue risk)"

    recs_text = "\n".join(f"- {r}" for r in recs)

    prompt = f"""
You are a Digital Health Coach. Combine ALL of the following sources:

1) User's habits and extracted numeric features (steps, sleep, water, calories, stress, age, BMI, resting heart rate).
2) ML-based fatigue risk probability: {ml_risk_text}.
3) Rule-based recommendations derived from their habits:
{recs_text}
4) RAG guideline context from WHO/CDC/NHS:
{context_text}

User's original question:
{user_question}

User's extracted features:
{user_context}

Using these, give a short, practical answer that:
- Explains their current situation (activity, sleep, stress, etc.).
- Mentions whether their fatigue risk is high, moderate, or low (if computed).
- Uses guidelines to justify targets (e.g., weekly activity, sleep range, hydration).
- Suggests 3–5 specific, realistic next steps.
- Avoids diagnosis and reminds them to consult a healthcare professional for medical concerns.
"""

    response = llm.invoke(prompt)
    final_answer = getattr(response, "content", str(response))

    return final_answer, top_chunks, ml_prob, recs, feats


In [ ]:
user_message = """
I'm 29 years old with a BMI around 28.5.
Last week I averaged about 4,500 steps a day, slept about 5–6 hours,
drank maybe 4 glasses of water, ate around 2600 calories.
My resting heart rate is about 78 and my stress feels like 7/10 most days.
What should I improve to reduce my fatigue and be healthier?
"""

answer, chunks, ml_prob, recs, feats = health_coach_agent_free_text(
    user_question=user_message,
    model=rf,
    top_k=4
)

print("Predicted high fatigue probability:", ml_prob)
print("\nExtracted features:", feats)
print("\nRecommendations (rule/ML-based):")
for r in recs:
    print("-", r)

print("\nFinal answer from LLM (combined):")
print(answer)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks(query: str, user_context: dict | None = None, top_k: int = 5):
    """
    Given a question and optional user context, return top_k most similar chunks
    from kb_index using cosine similarity over embeddings.
    """
    full_query = query
    if user_context is not None:
        ctx_parts = []
        age = user_context.get("age")
        bmi = user_context.get("bmi")

        if age is not None:
            ctx_parts.append(f"age {age}")
        if bmi is not None:
            # only format as float if not None
            ctx_parts.append(f"BMI {float(bmi):.1f}")

        # you can add more context if present
        avg_steps = user_context.get("avg_steps_7d")
        if avg_steps is not None:
            ctx_parts.append(f"avg steps last 7 days {avg_steps}")

        avg_sleep = user_context.get("avg_sleep_7d")
        if avg_sleep is not None:
            ctx_parts.append(f"avg sleep last 7 days {avg_sleep} hours")

        avg_stress = user_context.get("avg_stress_7d")
        if avg_stress is not None:
            ctx_parts.append(f"avg stress {avg_stress}/10")

        if ctx_parts:
            full_query = query + " | " + ", ".join(ctx_parts)

    # Encode query
    q_emb = model_emb.encode([full_query])

    # Stack chunk embeddings
    chunk_embs = np.stack(kb_index["embedding"].values)

    # Cosine similarity
    sims = cosine_similarity(q_emb, chunk_embs)[0]

    # Top-k indices
    top_idx = np.argsort(sims)[::-1][:top_k]

    return kb_index.iloc[top_idx].copy(), sims[top_idx]


In [ ]:
user_message = """
I’ve been feeling tired most days recently. My job keeps me sitting almost the whole day, and I usually snack a lot and drink very little water. I also sleep late because of using my phone too much at night. What should I do to improve my health?
"""

answer, chunks, ml_prob, recs, feats = health_coach_agent_free_text(
    user_question=user_message,
    model=rf,
    top_k=4
)

print("Predicted high fatigue probability:", ml_prob)
print("\nExtracted features:", feats)
print("\nRecommendations (rule/ML-based):")
for r in recs:
    print("-", r)

print("\nFinal answer from LLM (combined):")
print(answer)


In [ ]:
user_message = """
I'm working as a data analyst so I always sitting in the same position whole day and I'm getting fat everyday rise the body weight also what I can do for this give me proper nutrion plan and phsycal activity"""

answer, chunks, ml_prob, recs, feats = health_coach_agent_free_text(
    user_question=user_message,
    model=rf,
    top_k=4
)

print("Predicted high fatigue probability:", ml_prob)
print("\nExtracted features:", feats)
print("\nRecommendations (rule/ML-based):")
for r in recs:
    print("-", r)

print("\nFinal answer from LLM (combined):")
print(answer)


In [ ]:
pip install graphviz

In [ ]:
from graphviz import Digraph

dot = Digraph("HealthCoachArchitecture", format="png")
dot.attr(rankdir="LR", fontsize="10", fontname="Helvetica")

# ===== OFFLINE SUBGRAPH =====
with dot.subgraph(name="cluster_offline") as c:
    c.attr(label="Offline Preparation", style="filled", color="#e5e7eb")
    c.node("csv", "Synthetic CSV\nhealth_synthetic_200users_90days.csv", shape="folder")
    c.node("feat", "Feature Engineering\n(7-day rolling stats)", shape="box")
    c.node("rf_train", "Train RandomForest\n(high_fatigue_flag)", shape="box")
    c.node("rf_model", "Fatigue Risk Model\n(RandomForest)", shape="component")

    c.edge("csv", "feat")
    c.edge("feat", "rf_train")
    c.edge("rf_train", "rf_model")

    c.node("pdfs", "Guideline PDFs\n(WHO, Eatwell, PA, etc.)", shape="folder")
    c.node("pdf_text", "PDF → Text\n(pdfplumber)", shape="box")
    c.node("chunk", "Chunking\n(overlapping text chunks)", shape="box")
    c.node("embed", "Embeddings\n(SentenceTransformer)", shape="box")
    c.node("kb_index", "KB Index\nkb_index.parquet", shape="component")

    c.edge("pdfs", "pdf_text")
    c.edge("pdf_text", "chunk")
    c.edge("chunk", "embed")
    c.edge("embed", "kb_index")

# ===== ONLINE SUBGRAPH =====
with dot.subgraph(name="cluster_online") as c:
    c.attr(label="Online Inference (Streamlit App)", style="filled", color="#dbeafe")
    c.node("user", "User", shape="oval")
    c.node("ui", "Streamlit Chat UI", shape="box")
    c.node("agent", "health_coach_agent_free_text\n(Orchestrator)", shape="box", style="rounded")

    c.node("llm_parse", "Groq LLM\n(feature extraction)", shape="box")
    c.node("features", "Extracted features\n(steps, sleep, water, stress, age, BMI…)", shape="box")

    c.node("rf_use", "RandomForest\npredict_proba", shape="box")
    c.node("risk", "Fatigue Risk\n(probability)", shape="diamond")

    c.node("rules", "Rule-based engine\n(thresholds)", shape="box")
    c.node("recs", "Recommendations\n(ML + rules)", shape="box")

    c.node("retr", "Retriever\n(semantic search)", shape="box")
    c.node("chunks", "Top guideline chunks", shape="box")

    c.node("llm_answer", "Groq LLM\n(final coaching answer)", shape="box")
    c.node("resp", "Final response\n(coaching text)", shape="note")

    # Edges in online flow
    c.edge("user", "ui")
    c.edge("ui", "agent")

    c.edge("agent", "llm_parse", label="1. parse habits")
    c.edge("llm_parse", "features")

    c.edge("features", "rf_use")
    c.edge("rf_use", "risk")

    c.edge("features", "rules")
    c.edge("rules", "recs")

    c.edge("agent", "retr", label="2. RAG query")
    c.edge("retr", "kb_index")
    c.edge("kb_index", "chunks")

    c.edge("agent", "llm_answer", label="3. combine\nfeatures + risk + recs + chunks")
    c.edge("llm_answer", "resp")
    c.edge("resp", "ui")
    c.edge("ui", "user")

# Render to file
output_path = dot.render("health_coach_architecture", cleanup=True)
print("Diagram saved to:", output_path)
